# Data Loading

In [8]:
%%time

import kplex_dev
import data_splits
import warnings
import pandas as pd
import numpy as np
import ast
from sklearn.preprocessing import LabelEncoder


warnings.filterwarnings('ignore')

timeout_replacement = 1800  # max timeout replacement
excel_path = "kplex_ML_data.xlsx"

feature_cols = data_splits.feature_cols
algorithms = data_splits.algorithms


CPU times: total: 0 ns
Wall time: 0 ns


In [2]:
# Loading and preprocessing data
cat_df, dfs = kplex_dev.load_excel_data(excel_path, timeout_replacement)
kplex_dev.process_kcore_list_to_statistics(cat_df, keep_col=True)

# Selecting weight scheme
epsilon = 0.001 # need to set !!!
scheme = kplex_dev.W_TIME_MIN_X_W_LOG_TIME_RATIO # Change this and analyse!!

# try dense cross entrpy loss for NN
# take 10% of data out as unseen data and run triaing and test split on remaining 90%data

# Assigning weights
kplex_dev.assign_weights(cat_df, scheme, epsilon)

# kplex_dev.export_sorted(cat_df)

# Data reformatting
cat_df = kplex_dev.post_process_data(cat_df)
cat_df['kcore_list'] = cat_df['kcore_list'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# fastest_time = minimum across those columns
cat_df['fastest_time'] = cat_df[data_splits.algorithms].min(axis=1)

cat_df

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,...,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br,kcore_list,fastest_time
0,MANN-a81,20,3321,5506380,1658.0488,3317,3280,3280,1,3321.000000,...,3321,0.000000,306.046574,1459.5,1800.0,1468.1,1477.4,1504.7,"[(3280, 3321)]",1459.5
1,wikipedia_link_sr,20,1810,1637059,904.4525,1809,1798,1795,4,1806.250000,...,1803,2.861381,66.902739,22.6,18.0,737.7,740.4,738.1,"[(1795, 1810), (1796, 1808), (1797, 1804), (17...",18.0
2,scc_reality,20,1332,880733,661.2110,1331,1235,1230,6,1290.333333,...,1236,35.714921,47.684856,1800.0,1214.8,9.4,9.7,9.0,"[(1230, 1332), (1231, 1330), (1232, 1309), (12...",9.0
3,web-wikipedia_link_en13-all,20,432,93085,215.4745,431,427,427,1,432.000000,...,432,0.000000,184.662528,239.3,287.1,517.7,504.4,516.7,"[(427, 432)]",239.3
4,livejournal-links,20,442,94435,213.6538,441,371,369,3,412.333333,...,386,22.983086,0.269931,1.3,1.6,1.6,1.6,1.6,"[(369, 442), (370, 409), (371, 386)]",1.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,2,264,11792,44.6667,150,67,52,16,215.000000,...,125,56.656862,0.784308,2.0,0.6,0.5,0.5,2.4,"[(52, 264), (53, 262), (54, 261), (55, 260), (...",0.5
756,edit-ukwiki,2,281,14183,50.4733,280,65,40,26,235.384615,...,168,34.703222,0.437734,1.2,1.1,0.6,0.6,0.5,"[(40, 281), (41, 279), (42, 275), (43, 271), (...",0.5
757,johnson8-4-4,2,70,1855,26.5000,53,53,53,1,70.000000,...,70,0.000000,0.389785,0.8,0.9,1.1,1.1,0.3,"[(53, 70)]",0.3
758,hamming6-2,2,64,1824,28.5000,57,57,57,1,64.000000,...,64,0.000000,0.081093,0.2,0.3,0.2,0.2,0.3,"[(57, 64)]",0.2


# Training with LSTM

Simple LSTM network with a soft‑cross‑entropy loss.


### Model

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.metrics import classification_report

In [14]:
# kplex_dev.py

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted

# ——— 1) Soft Cross‐Entropy with Temperature ——————————————

class SoftCrossEntropyLoss(nn.Module):
    """
    Cross‐entropy loss that accepts either integer targets or soft‐probability targets.
    If `targets` is long dtype → uses standard CE; if float dtype & 2D → uses -sum(p * log_softmax).
    """
    def __init__(self, temperature: float = 1.0):
        super().__init__()
        self.temperature = temperature

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        scaled = logits / self.temperature
        if targets.dtype in (torch.int64, torch.long):
            # hard labels
            return F.cross_entropy(scaled, targets)
        else:
            # soft labels: expects shape (B, C), rows sum to ~1
            log_probs = F.log_softmax(scaled, dim=1)
            return -torch.sum(targets * log_probs, dim=1).mean()

# ——— 2) Custom Dataset to yield features + k_core sequences ——————
import ast
import numpy as np
import torch
from torch.utils.data import Dataset

class KCoreDataset(Dataset):
    def __init__(self, X, y, k_core_list, scaler):
        assert len(X) == len(y) == len(k_core_list), (
            f"X ({len(X)}), y ({len(y)}), and k_core_list ({len(k_core_list)}) must all have the same length"
        )
        # 1) Scale features once
        self.X = scaler.transform(X)
        self.y = np.array(y, dtype=int)
        
        # 2) Parse & cache sequences as numeric arrays
        self.seqs = []
        for entry in k_core_list:
            parsed = entry
            # If it's a string (including numpy.str_), un-wrap until we get a list
            while isinstance(parsed, str):
                parsed = ast.literal_eval(parsed)
            # Now parsed should be a list of (int, int) tuples
            arr = np.array(parsed, dtype=float)  # shape (L,2)
            self.seqs.append(arr)
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        features = torch.tensor(self.X[idx], dtype=torch.float32)
        target   = torch.tensor(self.y[idx], dtype=torch.long)
        seq_arr  = self.seqs[idx]                         # np.ndarray (L,2)
        seq      = torch.tensor(seq_arr, dtype=torch.float32)
        return features, seq, target


from torch.nn.utils.rnn import pad_sequence

# get a nice tensor with equal size, straightforward indexing, and LSTM just reads zeros on the padded steps.
def pad_collate(batch):
    """
    batch is a list of (features, seq, target) tuples.
    We pad the seqs along time‐axis so they all match the longest in this batch.
    """
    feats, seqs, tgts = zip(*batch)

    # Stack features and targets normally
    feats = torch.stack(feats, 0)
    tgts  = torch.stack(tgts,  0)

    # Pad the sequences (each is [L_i, 2]) to shape [L_max, 2]
    # pad_sequence wants a list of [L_i,2] tensors
    padded = pad_sequence(seqs, batch_first=True, padding_value=0.0)
    # padded.shape == (batch_size, L_max, 2)

    return feats, padded, tgts



# ——— 3) LSTM‐based classifier ——————————————————————————

class TorchLSTMClassifier(BaseEstimator, ClassifierMixin):
    """
    sklearn‐compatible PyTorch LSTM classifier that
    incrementally leaks k_core_list tuples over epochs.
    """
    def __init__(
        self,
        hidden_layers=(128,64),
        lstm_hidden=32,
        lr: float = 1e-2,
        batch_size: int = 64,
        epochs: int = 50,
        temperature: float = 1.0,
        device: str = None,
        random_state: int = 42
    ):
        self.hidden_layers = hidden_layers
        self.lstm_hidden   = lstm_hidden
        self.lr            = lr
        self.batch_size    = batch_size
        self.epochs        = epochs
        self.temperature   = temperature
        self.device        = device
        self.random_state  = random_state

    def _build_model(self, n_feat, n_classes):
        # feature MLP
        mlp_layers = []
        in_dim = n_feat
        for h in self.hidden_layers:
            mlp_layers += [nn.Linear(in_dim, h), nn.ReLU()]
            in_dim = h
        self.mlp = nn.Sequential(*mlp_layers)
        # LSTM encoder for k_core tuples
        self.lstm = nn.LSTM(input_size=2, hidden_size=self.lstm_hidden, batch_first=True)
        # final classifier
        tot_dim = in_dim + self.lstm_hidden
        self.classifier = nn.Linear(tot_dim, n_classes)
        return

    def fit(self, X, y, soft_labels: bool = False):
        """
        X: DataFrame with 'kcore_list' column + feature columns.
        y: either 1D array of ints (hard labels) or 2D array of soft-probabilities.
        soft_labels: whether y contains soft targets.
        """
        # extract and drop kcore_list
        k_core_list = [np.array(t) for t in X['kcore_list']]
        X_feat = X.drop('kcore_list', axis=1)

        # validate features + labels
        X_arr, _ = check_X_y(X_feat.values, y if not soft_labels else np.zeros(len(y)))
        n_samples, n_feat = X_arr.shape

        torch.manual_seed(self.random_state)
        self.device_ = torch.device(self.device or ('cuda' if torch.cuda.is_available() else 'cpu'))

        # scale
        self.scaler_ = StandardScaler().fit(X_arr)
        X_scaled   = self.scaler_.transform(X_arr)

        # prepare targets
        if soft_labels:
            y_arr = np.asarray(y, dtype=float)
            y_tensor = torch.tensor(y_arr, dtype=torch.float32, device=self.device_)
            n_classes = y_arr.shape[1]
        else:
            y_arr = np.asarray(y)
            self.classes_ = np.unique(y_arr)
            n_classes = len(self.classes_)
            self.class_to_idx_ = {c:i for i,c in enumerate(self.classes_)}
            y_idx = np.vectorize(self.class_to_idx_.get)(y_arr)
            y_tensor = torch.tensor(y_idx, dtype=torch.long, device=self.device_)

        # build model & optimizer
        self._build_model(n_feat, n_classes)
        self.mlp.to(self.device_); self.lstm.to(self.device_); self.classifier.to(self.device_)
        optimizer = optim.SGD(
            list(self.mlp.parameters()) +
            list(self.lstm.parameters()) +
            list(self.classifier.parameters()),
            lr=self.lr
        )
        criterion = SoftCrossEntropyLoss(self.temperature)

        # dataset & loader
        ds = KCoreDataset(X_arr, y_arr, k_core_list, self.scaler_)
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=True, collate_fn=pad_collate)

        # training
        for epoch in range(1, self.epochs+1):
            self.mlp.train(); self.lstm.train(); self.classifier.train()
            for feats, seq, targets in loader:
                feats, seq = feats.to(self.device_), seq.to(self.device_)
                if soft_labels:
                    tgt = targets.to(self.device_, dtype=torch.float32)
                    # tgt = torch.tensor(y_arr, dtype=torch.float32, device=self.device_)
                else:
                    tgt = targets.to(self.device_)
                optimizer.zero_grad()
                # leak k‐core tuples
                L = min(epoch, seq.size(1))
                truncated = seq[:, :L, :]
                _, (h_n, _) = self.lstm(truncated)
                lstm_feat = h_n[-1]
                mlp_feat  = self.mlp(feats)
                combined  = torch.cat([mlp_feat, lstm_feat], dim=1)
                logits    = self.classifier(combined)
                loss      = criterion(logits, tgt)
                loss.backward()
                optimizer.step()
        return self

    def predict(self, X):
        check_is_fitted(self, ['mlp','lstm','classifier','scaler_'])
        k_core_list = [np.array(t) for t in X['kcore_list']]
        X_feat = X.drop('kcore_list', axis=1).values
        X_scaled = self.scaler_.transform(X_feat)
        ds = KCoreDataset(X_scaled, np.zeros(len(X_scaled)), k_core_list, self.scaler_)
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=False, collate_fn=pad_collate)

        preds = []
        self.mlp.eval(); self.lstm.eval(); self.classifier.eval()
        with torch.no_grad():
            for feats, seq, _ in loader:
                feats, seq = feats.to(self.device_), seq.to(self.device_)
                _, (h_n, _) = self.lstm(seq)
                lstm_feat = h_n[-1]
                mlp_feat  = self.mlp(feats)
                combined  = torch.cat([mlp_feat, lstm_feat], dim=1)
                logits    = self.classifier(combined)
                preds.append(logits.argmax(dim=1).cpu().numpy())
        preds = np.concatenate(preds)
        if hasattr(self, 'class_to_idx_'):
            inv_map = {i:c for c,i in self.class_to_idx_.items()}
            return np.vectorize(inv_map.get)(preds)
        return preds

    def predict_proba(self, X):
        check_is_fitted(self, ['mlp','lstm','classifier','scaler_'])
        k_core_list = [np.array(t) for t in X['kcore_list']]
        X_feat = X.drop('kcore_list', axis=1).values
        X_scaled = self.scaler_.transform(X_feat)
        ds = KCoreDataset(X_scaled, np.zeros(len(X_scaled)), k_core_list, self.scaler_)
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=False, collate_fn=pad_collate)

        probs = []
        self.mlp.eval(); self.lstm.eval(); self.classifier.eval()
        with torch.no_grad():
            for feats, seq, _ in loader:
                feats, seq = feats.to(self.device_), seq.to(self.device_)
                _, (h_n, _) = self.lstm(seq)
                lstm_feat = h_n[-1]
                mlp_feat  = self.mlp(feats)
                combined  = torch.cat([mlp_feat, lstm_feat], dim=1)
                logits    = self.classifier(combined) / self.temperature
                probs.append(F.softmax(logits, dim=1).cpu().numpy())
        return np.vstack(probs)

In [5]:
def expand_with_ties_and_prepare(
    df,
    weight_col='weight',
    target_col_name='target',
    for_nn=False,
    temprature=1.0,
    tie_tolerance: float = 0.0   # 0% default
):
    """
    Given a DataFrame with multiple algorithm runtimes,
    expand rows to duplicate ties (within tolerance) and adjust weights,
    then return X, hard y, soft y, and sample weights.

    Returns
    -------
    X_out        : pd.DataFrame
    y_out        : pd.Series         # hard labels (one per expanded row)
    y_soft_out   : np.ndarray        # soft targets (same shape as X_out × n_algos)
    weights_out  : pd.Series
    """
    expanded_rows = []
    hard_labels   = []
    soft_labels   = []
    weights_list  = []

    # precompute feature set & number of algorithms
    n_alg = len(algorithms)

    for _, row in df.iterrows():
        # 1) raw runtimes and threshold
        runtimes = np.array([row[alg] for alg in algorithms], dtype=float)
        min_rt    = runtimes.min()
        thresh    = (1 + tie_tolerance) * min_rt

        # 2) hard‐tie set
        hard_set = [alg for alg, rt in zip(algorithms, runtimes) if rt <= thresh]

        # 3) soft‐target over *all* algos = softmax(1/runtime)
        inv = 1.0 / (runtimes + 1e-8)
        ex  = np.exp(inv/temprature)
        soft = ex / ex.sum()

        # 4) one expanded row per hard‐tie
        for alg in hard_set:
            if 'kcore_list' in row:
                new_row = row[feature_cols + ['kcore_list']].copy()
            elif for_nn:
                new_row = row[feature_cols + ['fastest_time']].copy()
            else:
                new_row = row[feature_cols].copy()

            expanded_rows.append(new_row)
            hard_labels.append(alg)
            soft_labels.append(soft)   # same soft vector for each copy
            weights_list.append(row[weight_col] / len(hard_set))

    expanded_df = pd.DataFrame(expanded_rows)

    X_out       = expanded_df.drop([target_col_name, weight_col], axis=1, errors='ignore')
    y_out       = pd.Series(hard_labels,  name=target_col_name)
    y_soft_out  = np.vstack(soft_labels)          # shape (n_expanded, n_alg)
    weights_out = pd.Series(weights_list, name=weight_col)

    return X_out, y_out, y_soft_out, weights_out


## Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


### **Balanced train-test split based on Y**

In [6]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (604, 22)
Test set shape:  (156, 22)


In [ ]:
from sklearn.preprocessing import LabelEncoder

X_train_, y_train_hard, y_train_soft, weights_out = expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.0,
    temprature=1  # For regulating the softmax labels
)

from sklearn.preprocessing import LabelEncoder
# Label‐encode the string targets into integers 0…4 ---
le = LabelEncoder()
y_train_hard = le.fit_transform(y_train_hard)     # e.g. ['AltRB','UMKP',…] → [0,3,…]


print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [ ]:

# Instantiate and train
# Before calling fit():
clf_lstm_holdout_strat_y = TorchLSTMClassifier(hidden_layers=(128, 64, 32), lstm_hidden=32, epochs=500, lr= 0.01)

clf_lstm_holdout_strat_y.fit(X_train_, y_train_soft, soft_labels=True)



TorchLSTMClassifier(hidden_layers=(128, 64, 32))

#### Inference on 20% dataset

In [19]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_lstm_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    kcore_list = True,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LSTM_results_soft_labels/tie_tol/00/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LSTM_results_soft_labels/tie_tol/00/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 32.05%

Precision by Algorithm:
  UMKP: 26.67% (15 predictions)
  AltRB: 0.00% (0 predictions)
  kPEX: 0.00% (0 predictions)
  Pivot_Br: 0.00% (0 predictions)
  S_Br: 32.62% (141 predictions)

Recall by Algorithm:
  UMKP: 35.00% (7/20 cases)
  AltRB: 17.39% (4/23 cases)
  kPEX: 23.29% (17/73 cases)
  Pivot_Br: 37.78% (17/45 cases)
  S_Br: 92.16% (47/51 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 59839.50
  Total Optimal Time   = 16592.90
  Time Ratio (pred/opt)= 3.606
  Avg Overhead/case ((pred - opt) / opt)    = 11727.83%


Adding prediction probabilities...

Saving full predictions to './LSTM_results_soft_labels/tie_tol/00/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 106 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                         dat

#### Re-evalute on entire dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_lstm_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    kcore_list = True,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LSTM_results_soft_labels/tie_tol/00/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LSTM_results_soft_labels/tie_tol/00/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 49.87%

Precision by Algorithm:
  UMKP: 25.00% (8 predictions)
  AltRB: 49.26% (136 predictions)
  kPEX: 58.65% (237 predictions)
  Pivot_Br: 39.68% (247 predictions)
  S_Br: 55.30% (132 predictions)

Recall by Algorithm:
  UMKP: 36.00% (36/100 cases)
  AltRB: 73.28% (85/116 cases)
  kPEX: 63.36% (230/363 cases)
  Pivot_Br: 67.70% (153/226 cases)
  S_Br: 51.56% (132/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 99232.10
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.484
  Avg Overhead/case ((pred - opt) / opt)    = 289.29%


Adding prediction probabilities...

Saving full predictions to './LSTM_results/tie_tol/50/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 381 wrong predictions with time metrics

T

Additional time metrics:

- Total prediction time: sum of all runtimes for the algorithms our model actually chose, across every test instance
- Total Optimal Time:  sum of the runtime of the best (fastest) algorithm for each instance.
- Time Ratio (Predicted / Optimal)
- Average Overhead per Case: (predicted_time – optimal_time) / optimal_time + epsilon(0.001)

### **Balanced train-test split based on Y and K**

In [48]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (612, 22)
Test set shape:  (148, 22)


In [ ]:
from sklearn.preprocessing import LabelEncoder

X_train_, y_train_hard, y_train_soft, weights_out = expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.0,
    temprature=1 # for regulating the strength of soft labels
)

from sklearn.preprocessing import LabelEncoder
# Label‐encode the string targets into integers 0…4 ---
le = LabelEncoder()
y_train_hard = le.fit_transform(y_train_hard)     # e.g. ['AltRB','UMKP',…] → [0,3,…]


print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [ ]:
# Instantiate and train
# Before calling fit():
clf_lstm_holdout_strat_y_K = TorchLSTMClassifier(hidden_layers=(128, 64, 32), lstm_hidden=32, epochs=1000, lr= 0.01)

# clf_lstm_holdout_strat_y_K.fit(X_train_, y_train_)
clf_lstm_holdout_strat_y.fit(X_train_, y_train_soft, soft_labels=True)



TorchLSTMClassifier(epochs=1000, hidden_layers=(128, 64, 32))

#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=clf_lstm_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LSTM_results_soft_labels/tie_tol/00/split_and_expand',
    kcore_list = True,
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LSTM_results_soft_labels/tie_tol/00/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 51.19%

Precision by Algorithm:
  UMKP: 50.00% (4 predictions)
  AltRB: 55.38% (65 predictions)
  kPEX: 64.29% (196 predictions)
  Pivot_Br: 27.27% (77 predictions)
  S_Br: 38.46% (78 predictions)

Recall by Algorithm:
  UMKP: 49.06% (26/53 cases)
  AltRB: 73.91% (51/69 cases)
  kPEX: 68.16% (152/223 cases)
  Pivot_Br: 60.65% (94/155 cases)
  S_Br: 54.11% (79/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 55153.20
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.189
  Avg Overhead/case ((pred - opt) / opt)    = 124.83%


Adding prediction probabilities...

Saving full predictions to './LSTM_results/tie_tol/50/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 205 wrong predictions with time metrics

Top 5 hea

#### Re-evalute on entire dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_lstm_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LSTM_results_soft_labels/tie_tol/00/split_and_expand',
    kcore_list = True,
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LSTM_results_soft_labels/tie_tol/00/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.92%

Precision by Algorithm:
  UMKP: 50.00% (14 predictions)
  AltRB: 52.07% (121 predictions)
  kPEX: 60.94% (361 predictions)
  Pivot_Br: 23.02% (126 predictions)
  S_Br: 49.28% (138 predictions)

Recall by Algorithm:
  UMKP: 39.00% (39/100 cases)
  AltRB: 72.41% (84/116 cases)
  kPEX: 69.42% (252/363 cases)
  Pivot_Br: 57.08% (129/226 cases)
  S_Br: 54.30% (139/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 96591.70
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.444
  Avg Overhead/case ((pred - opt) / opt)    = 257.20%


Adding prediction probabilities...

Saving full predictions to './LSTM_results/tie_tol/50/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 373 wrong predictions with time metrics